In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

#from pathlib import Path

#path importation
DATA_PATH = r"C:\Users\HP FOLIO\Downloads\bitcoin-historical-data (1).csv"

print("The path has been loaded and ready to be read into data-frame.....")

print("The data cleaning has started...")

#1. Read the data into data frame
df = pd.read_csv(
    DATA_PATH,
    na_values=["$", "?", "NA", "N/A", "", "NaN", "nan"],
    keep_default_na=True
)

print("The BTC historical data csv file has been read into data-frame and ready for exploration...")

df.info()

#2. Replace NAN with UNKNOWN

#Note: Since "Vol Chg" and "MCap Chg" are the only columns where there is missing values
#Note: we will iterate through the missing columns("Vol Chg" and "MCap Chg") with missing values instead of iterating through the whole columns and fill it will UNKNOWN

missing_columns = ["Vol Chg", "MCap Chg"]

df.head(50)

for col in missing_columns:
    if col in df.columns:
        df[col] = df[col].fillna("UNKNOWN")
    else:
        print(f"The column {col} is is not present in the data frame columns")

df.info()

print("The missing values have been replaced by the string UNKNOWN")

#3. Dropping of Date Columns since we don't need it for the model training

column_to_drop = ["Date", "Vol Chg", "MCap Chg"]

for col in column_to_drop:
    df = df.drop(col, axis=1)
#df = df.drop("Date", axis=1)


print("The date column has been dropped from the date-frame")

#4. We want to remove the $ from all the columns where the symbol is used

print("Defining columns with $ symbol.....")

dollar_columns = ["Open", "High", "Low", "Avg", "Close", "Volume", "Market Cap"]

for col in dollar_columns:
    if col in df.columns:
        #df[col] = df[col].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
        df[col] = df[col].str.replace({"$": "", ",": ""})
    else:
        print(f"The column {col} cannot be found")

print("$ and , has been removed")

#5. Removal of whitespaces.

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

print("The whitespaces have been stripped, Feel free to add naked😂")

#6. Removal of last line from the data-frame because it contains incomplete data and we don't want it to corrupt our model during training
#df = df.iloc[0:]


df.info()


In [118]:
#6. Initialization of model training

print("Intialization of model training....")

df.info()

#First let's convert the type of each column back to float because they have been converted to str during stripping
for col in df.columns:
    df[col] = df[col].astype(float)

#Note: Row 2 be our "y" and make row 3 - 30 "X".  We are doing this because
#1. We want to train our data with a large data
#2. After the model has been trained we want to test the model with row 0. i.e. we will input row 1 into the model to predict row 0 and cross the predicted values with actual values of row 1

#our_sample = df.iloc[0:3]

# Predict TOMORROW'S close
df["Target"] = df["Close"].shift(-1)

# Remove last row (no future target)
df = df.dropna()

print(f" The dataframe is \n{df.head(50)}")

# =========================
# 4. Features (X) and Target (y)
# =========================

y = df["Target"]
X = df.drop(["Close", "Target"], axis=1, errors="ignore") # errors="ignore" -- “If the column does not exist, do not raise an error.”

#df_copy = df.copy()

# This still contains CLOSE column
#input = df_copy.iloc[[0]] # This keeps it as a DataFrame.

# Removal of close column
#input_data = input.drop(["Close"], axis=1)

#df_copy = df_copy.iloc[1:]

#y = df_copy["Close"]
#y = y.apply(pd.to_numeric, errors='coerce')
print(f" Our y is\n{y}")
print(f"its length is {len(y)}")

#X = X.apply(pd.to_numeric, errors='coerce')

#X = df_copy.drop(["Close"], axis=1)
#X = X.apply(pd.to_numeric, errors='coerce')
print(f"Our X is\n{X}")
print(f"its length is {len(y)}")

print("Our X and y has been defined")



Intialization of model training....
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Open        32 non-null     str  
 1   High        32 non-null     str  
 2   Low         32 non-null     str  
 3   Avg         32 non-null     str  
 4   Close       32 non-null     str  
 5   Chg         32 non-null     str  
 6   Volume      32 non-null     str  
 7   Market Cap  32 non-null     str  
dtypes: str(8)
memory usage: 2.1 KB
 The dataframe is 
       Open     High      Low      Avg    Close      Chg        Volume  \
0   74498.0  75884.0  73963.0  74618.0  74130.0 -0.49300  2.964760e+10   
1   70739.0  74807.0  70625.0  72679.0  74546.0  5.38000  2.941988e+10   
2   73062.0  73123.0  70618.0  71883.0  70728.0 -3.20000  2.149231e+10   
3   72940.0  73719.0  72637.0  73088.0  73057.0  0.16000  2.049370e+10   
4   71780.0  73358.0  71463.0  72391.0  72965.0  1.65000  

In [119]:
#7. Model training

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# =========================
# 5. Time-Series Split (NO SHUFFLE)
# =========================

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Reversal of dataframe
#df = df.iloc[::-1].reset_index(drop=True)

split_index = int(len(df) * 0.8)

X_train = X[:split_index]
X_test = X[split_index:]


y_train = y[:split_index]
y_test = y[split_index:]

# =========================
# 6. Train Model
# =========================

model = RandomForestRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

print("Model training complete!")

# =========================
# 7. Evaluate Model
# =========================

predictions = model.predict(X_test)

mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Squared Error: {mse:.2f} (Lower is better)")
print(f"R-squared Score: {r2:.2f} (Closer to 1.0 is better)")

#model = LinearRegression()

#model.fit(X_train, y_train)
#print("Model training complete!")

#print(f"The X_test is \n{X_test}")

#predictions = model.predict(X_test)

#mse = mean_squared_error(y_test, predictions)
#r2 = r2_score(y_test, predictions)


Model training complete!
Mean Squared Error: 437123.57 (Lower is better)
R-squared Score: 0.85 (Closer to 1.0 is better)


In [120]:
# =========================
# 8. Predict Next Day
# =========================

# Take latest row as input (DataFrame, not Series)
# iloc means: Select rows/columns using integer positions.
# -1 means the last row
# input_data = X.iloc[[-1]] #means: Select the last row of X as a DataFrame
input_data = X.iloc[[0]]

print(f"The Input data is \n{input_data}")

result = model.predict(input_data)

print(f"Predicted NEXT BTC Close: {result[0]}")

#result = model.predict(input_data)

#print(f"The predicted BTC data is {result}")

The Input data is 
      Open     High      Low      Avg    Chg        Volume    Market Cap
0  74498.0  75884.0  73963.0  74618.0 -0.493  2.964760e+10  1.483779e+12
Predicted NEXT BTC Close: 73916.5175
